In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/competitions/ieee-fraud-detection/sample_submission.csv
/kaggle/input/competitions/ieee-fraud-detection/test_identity.csv
/kaggle/input/competitions/ieee-fraud-detection/train_identity.csv
/kaggle/input/competitions/ieee-fraud-detection/test_transaction.csv
/kaggle/input/competitions/ieee-fraud-detection/train_transaction.csv


In [2]:
!pip install dagshub
!pip install mlflow

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 273.1/273.1 kB 6.4 MB/s eta 0:00:00ta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.2/68.2 kB 5.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.3/207.3 kB 16.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 89.9/89.9 kB 7.2 MB/s eta 0:00:00
  Attempting uninstall: dacite
    Found existing installation: dacite 1.9.2
    Uninstalling dacite-1.9.2:
      Successfully uninstalled dacite-1.9.2
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
ydata-profiling 4.18.1 requires dacite<2,>=1.9, but you have dacite 1.6.0 which is incompatible.
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.2/49.2 kB 2.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.0/50.0 kB 2.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 2.9 MB/s eta 0:00:00
   ━━━━━━

In [3]:
import dagshub
dagshub.init(repo_owner='icosahedron31', repo_name='IEEE-CIS-Fraud-Detection', mlflow=True)



❗❗❗ AUTHORIZATION REQUIRED ❗❗❗



Open the following link in your browser to authorize the client:
https://dagshub.com/login/oauth/authorize?state=3d98973d-cf58-491f-8d03-792652ee5664&client_id=32b60ba385aa7cecf24046d8195a71c07dd345d9657977863b52e7748e0f0f28&middleman_request_id=db94f94183745962221468dd8a8018892d52fb4e6c167addf64273c4393ee237




Output()

Accessing as icosahedron31

Initialized MLflow to track repo "icosahedron31/IEEE-CIS-Fraud-Detection"

Repository icosahedron31/IEEE-CIS-Fraud-Detection initialized!

# Reading Data

In [4]:
df_identity = pd.read_csv("/kaggle/input/competitions/ieee-fraud-detection/train_identity.csv")
df_identity.head(20)

,TransactionID,id_01,id_02,id_03,id_04,id_05,id_06,id_07,id_08,id_09,...,id_31,id_32,id_33,id_34,id_35,id_36,id_37,id_38,DeviceType,DeviceInfo
0,2987004,0.0,70787.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,samsung browser 6.2,32.0,2220x1080,match_status:2,T,F,T,T,mobile,SAMSUNG SM-G892A Build/NRD90M
1,2987008,-5.0,98945.0,NaN,NaN,0.0,-5.0,NaN,NaN,NaN,...,mobile safari 11.0,32.0,1334x750,match_status:1,T,F,F,T,mobile,iOS Device
2,2987010,-5.0,191631.0,0.0,0.0,0.0,0.0,NaN,NaN,0.0,...,chrome 62.0,NaN,NaN,NaN,F,F,T,T,desktop,Windows
3,2987011,-5.0,221832.0,NaN,NaN,0.0,-6.0,NaN,NaN,NaN,...,chrome 62.0,NaN,NaN,NaN,F,F,T,T,desktop,NaN
4,2987016,0.0,7460.0,0.0,0.0,1.0,0.0,NaN,NaN,0.0,...,chrome 62.0,24.0,1280x800,match_status:2,T,F,T,T,desktop,MacOS
5,2987017,-5.0,61141.0,3.0,0.0,3.0,0.0,NaN,NaN,3.0,...,chrome 62.0,24.0,1366x768,match_status:2,T,F,T,T,desktop,Windows
6,2987022,-15.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,2987038,0.0,31964.0,0.0,0.0,0.0,-10.0,NaN,NaN,0.0,...,chrome 62.0,32.0,1920x1080,match_status:2,T,F,T,T,mobile,NaN
8,2987040,-10.0,116098.0,0.0,0.0,0.0,0.0,NaN,NaN,0.0,...,chrome 62.0,NaN,NaN,NaN,F,F,T,T,desktop,Windows
9,2987048,-5.0,257037.0,NaN,NaN,0.0,0.0,NaN,NaN,NaN,...,chrome 62.0,NaN,NaN,NaN,F,F,T,T,desktop,Windows


In [5]:
df_identity.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 144233 entries, 0 to 144232
Data columns (total 41 columns):
 #   Column         Non-Null Count   Dtype  
---  ------         --------------   -----  
 0   TransactionID  144233 non-null  int64  
 1   id_01          144233 non-null  float64
 2   id_02          140872 non-null  float64
 3   id_03          66324 non-null   float64
 4   id_04          66324 non-null   float64
 5   id_05          136865 non-null  float64
 6   id_06          136865 non-null  float64
 7   id_07          5155 non-null    float64
 8   id_08          5155 non-null    float64
 9   id_09          74926 non-null   float64
 10  id_10          74926 non-null   float64
 11  id_11          140978 non-null  float64
 12  id_12          144233 non-null  object 
 13  id_13          127320 non-null  float64
 14  id_14          80044 non-null   float64
 15  id_15          140985 non-null  object 
 16  id_16          129340 non-null  object 
 17  id_17          139369 non-nul

In [6]:
df_transaction = pd.read_csv("/kaggle/input/competitions/ieee-fraud-detection/train_transaction.csv")

In [7]:
df_transaction.info(), df_transaction.describe()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 590540 entries, 0 to 590539
Columns: 394 entries, TransactionID to V339
dtypes: float64(376), int64(4), object(14)
memory usage: 1.7+ GB


(None,
        TransactionID        isFraud  TransactionDT  TransactionAmt  \
 count   5.905400e+05  590540.000000   5.905400e+05   590540.000000   
 mean    3.282270e+06       0.034990   7.372311e+06      135.027176   
 std     1.704744e+05       0.183755   4.617224e+06      239.162522   
 min     2.987000e+06       0.000000   8.640000e+04        0.251000   
 25%     3.134635e+06       0.000000   3.027058e+06       43.321000   
 50%     3.282270e+06       0.000000   7.306528e+06       68.769000   
 75%     3.429904e+06       0.000000   1.124662e+07      125.000000   
 max     3.577539e+06       1.000000   1.581113e+07    31937.391000   
 
                card1          card2          card3          card5  \
 count  590540.000000  581607.000000  588975.000000  586281.000000   
 mean     9898.734658     362.555488     153.194925     199.278897   
 std      4901.170153     157.793246      11.336444      41.244453   
 min      1000.000000     100.000000     100.000000     100.000000   
 2

In [8]:
df_transaction.head()

,TransactionID,isFraud,TransactionDT,TransactionAmt,ProductCD,card1,card2,card3,card4,card5,...,V330,V331,V332,V333,V334,V335,V336,V337,V338,V339
0,2987000,0,86400,68.5,W,13926,NaN,150.0,discover,142.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2987001,0,86401,29.0,W,2755,404.0,150.0,mastercard,102.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2987002,0,86469,59.0,W,4663,490.0,150.0,visa,166.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,2987003,0,86499,50.0,W,18132,567.0,150.0,mastercard,117.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,2987004,0,86506,50.0,H,4497,514.0,150.0,mastercard,102.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [9]:
df_train = df_transaction.merge(df_identity, on='TransactionID', how='left')
df_train.columns

Index(['TransactionID', 'isFraud', 'TransactionDT', 'TransactionAmt',
       'ProductCD', 'card1', 'card2', 'card3', 'card4', 'card5',
       ...
       'id_31', 'id_32', 'id_33', 'id_34', 'id_35', 'id_36', 'id_37', 'id_38',
       'DeviceType', 'DeviceInfo'],
      dtype='object', length=434)

In [10]:
df_train.head(20)
df_train['isFraud']

0         0
1         0
2         0
3         0
4         0
         ..
590535    0
590536    0
590537    0
590538    0
590539    0
Name: isFraud, Length: 590540, dtype: int64

In [11]:
X = df_train.drop(columns=['isFraud'])
y = df_train['isFraud']
X.shape, y.shape

((590540, 433), (590540,))

# Train/test Split


In [12]:
from sklearn.model_selection import train_test_split



X_train, X_val, y_train, y_val = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42
)

X_train.shape, X_val.shape

((472432, 433), (118108, 433))

In [13]:
import mlflow
import mlflow.sklearn
import matplotlib.pyplot as plt
import numpy as np
from sklearn.metrics import (
    roc_auc_score,
    accuracy_score,
    recall_score,
    precision_score,
    f1_score,
    roc_curve,
    precision_recall_curve,
    confusion_matrix,
    ConfusionMatrixDisplay
)

def run_experiment_rf(
    model,
    run_name,
    experiment_name,
    X_train,
    X_val,
    y_train,
    y_val
):
    mlflow.set_experiment(experiment_name)
    with mlflow.start_run(run_name=run_name):
        model.fit(X_train, y_train)

        probs       = model.predict_proba(X_val)[:, 1]
        preds       = model.predict(X_val)
        probs_train = model.predict_proba(X_train)[:, 1]

        auc       = roc_auc_score(y_val, probs)
        train_auc = roc_auc_score(y_train, probs_train)
        accuracy  = accuracy_score(y_val, preds)
        recall    = recall_score(y_val, preds)
        precision = precision_score(y_val, preds, zero_division=0)
        f1        = f1_score(y_val, preds, zero_division=0)

        fraud_mask    = (y_val == 1)
        nonfraud_mask = (y_val == 0)

        mlflow.log_metrics({
            "train_auc":          train_auc,
            "auc":                auc,
            "accuracy":           accuracy,
            "recall":             recall,
            "precision":          precision,
            "f1":                 f1,
            "fraud_mean_prob":    probs[fraud_mask].mean(),
            "nonfraud_mean_prob": probs[nonfraud_mask].mean()
        })

        if hasattr(model, "get_params"):
            params = {k: str(v) for k, v in model.get_params().items()}
            mlflow.log_params(params)

        fpr, tpr, _ = roc_curve(y_val, probs)
        plt.figure()
        plt.plot(fpr, tpr, label=f"AUC = {auc:.4f}")
        plt.plot([0, 1], [0, 1], linestyle="--")
        plt.xlabel("False Positive Rate")
        plt.ylabel("True Positive Rate")
        plt.title("ROC Curve - Random Forest")
        plt.legend()
        mlflow.log_figure(plt.gcf(), "roc_curve.png")
        plt.close()

        precision_vals, recall_vals, _ = precision_recall_curve(y_val, probs)
        plt.figure()
        plt.plot(recall_vals, precision_vals)
        plt.xlabel("Recall")
        plt.ylabel("Precision")
        plt.title("Precision-Recall Curve - Random Forest")
        mlflow.log_figure(plt.gcf(), "pr_curve.png")
        plt.close()

        cm = confusion_matrix(y_val, preds)
        plt.figure()
        ConfusionMatrixDisplay(cm).plot()
        plt.title("Confusion Matrix - Random Forest")
        mlflow.log_figure(plt.gcf(), "confusion_matrix.png")
        plt.close()

        plt.figure()
        plt.hist(probs[nonfraud_mask], bins=50, alpha=0.5, label="non_fraud")
        plt.hist(probs[fraud_mask],    bins=50, alpha=0.5, label="fraud")
        plt.xlabel("Predicted Probability")
        plt.ylabel("Count")
        plt.title("Prediction Score Distribution - Random Forest")
        plt.legend()
        mlflow.log_figure(plt.gcf(), "score_distribution.png")
        plt.close()

        if hasattr(model, "feature_importances_"):
            importances   = model.feature_importances_
            feature_names = X_train.columns.tolist() if hasattr(X_train, "columns") else [f"feature_{i}" for i in range(X_train.shape[1])]
            indices = np.argsort(importances)[::-1][:20]
            plt.figure(figsize=(10, 6))
            plt.barh(
                [feature_names[i] for i in indices][::-1],
                importances[indices][::-1]
            )
            plt.xlabel("Importance")
            plt.title("Top 20 Feature Importances - Random Forest")
            plt.tight_layout()
            mlflow.log_figure(plt.gcf(), "feature_importances.png")
            plt.close()

        mlflow.sklearn.log_model(model, "model")

        return {
            "train_auc": train_auc,
            "auc":       auc,
            "accuracy":  accuracy,
            "precision": precision,
            "recall":    recall,
            "f1":        f1
        }

# Preprocessing

In [15]:
import pandas as pd
from sklearn.base import BaseEstimator, TransformerMixin


class MissingValueFilter(BaseEstimator, TransformerMixin):

    def __init__(
        self,
        column_threshold=0.9,
        row_threshold=None
    ):
        self.column_threshold = column_threshold
        self.row_threshold = row_threshold

    def fit(self, X, y=None):

        missing_ratio = X.isnull().mean()

        self.columns_to_keep_ = missing_ratio[
            missing_ratio <= self.column_threshold
        ].index.tolist()

        return self

    def transform(self, X):

        X = X.copy()

        X = X[self.columns_to_keep_]

        if self.row_threshold is not None:

            row_missing_ratio = X.isnull().mean(axis=1)

            X = X[
                row_missing_ratio <= self.row_threshold
            ]

        return X

In [16]:
import pandas as pd
from sklearn.base import BaseEstimator, TransformerMixin


class NAPreprocessor(BaseEstimator, TransformerMixin):

    def __init__(
        self,
        numeric_strategy="median",
        categorical_strategy="mode"
    ):
        self.numeric_strategy = numeric_strategy
        self.categorical_strategy = categorical_strategy

    def fit(self, X, y=None):
        X = X.copy()

        self.fill_values_ = {}

        for col in X.columns:

            # Numeric columns
            if pd.api.types.is_numeric_dtype(X[col]):

                if self.numeric_strategy == "mean":
                    value = X[col].mean()

                elif self.numeric_strategy == "median":
                    value = X[col].median()

             

            # Categorical columns
            else:

                if self.categorical_strategy == "mode":

                    mode_vals = X[col].mode()

                    if len(mode_vals) > 0:
                        value = mode_vals.iloc[0]
                    else:
                        value = "missing"

                elif self.categorical_strategy == "constant":
                    value = "missing"


            self.fill_values_[col] = value

        return self

    def transform(self, X):
        X = X.copy()

        for col, fill_value in self.fill_values_.items():

            if col in X.columns:
                X[col] = X[col].fillna(fill_value)

        return X

In [18]:
import pandas as pd
import numpy as np
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.preprocessing import OrdinalEncoder

class FraudPreprocessor(BaseEstimator, TransformerMixin):
    def __init__(self):
        self.cat_cols = None
        self.num_cols = None
        self.encoder = None
        self.medians = None

    def fit(self, X, y=None):
        X = X.copy()
        self.cat_cols = X.select_dtypes(include=["object", "category"]).columns.tolist()
        self.num_cols = X.select_dtypes(include=[np.number, "bool"]).columns.tolist()
        self.medians = X[self.num_cols].median()
        if self.cat_cols:
            X[self.cat_cols] = X[self.cat_cols].fillna("missing")
            self.encoder = OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1)
            self.encoder.fit(X[self.cat_cols])
        return self

    def transform(self, X):
        X = X.copy()
        if self.cat_cols:
            X[self.cat_cols] = X[self.cat_cols].fillna("missing")
            if self.encoder is not None:
                X[self.cat_cols] = self.encoder.transform(X[self.cat_cols])
        for col in self.num_cols:
            if col in X.columns:
                X[col] = X[col].fillna(self.medians[col])
        for col in X.columns:
            X[col] = pd.to_numeric(X[col], errors="coerce")
        X = X.fillna(0)
        return X

In [21]:
import pandas as pd
import numpy as np

class WOEEncoder(BaseEstimator, TransformerMixin):
    def __init__(self, cols=None, smoothing=0.5):
        self.cols = cols
        self.smoothing = smoothing
        self.woe_maps = {}

    def fit(self, X, y):
        X = X.copy()
        cols = self.cols or X.select_dtypes(include=["object", "category"]).columns.tolist()
        
        total_events = y.sum()
        total_non_events = (1 - y).sum()

        for col in cols:
            tmp = pd.DataFrame({"col": X[col], "target": y.values})
            stats = tmp.groupby("col")["target"].agg(["sum", "count"])
            stats.columns = ["events", "count"]
            stats["non_events"] = stats["count"] - stats["events"]

            # smoothing to avoid log(0)
            stats["dist_events"]     = (stats["events"] + self.smoothing) / (total_events + self.smoothing)
            stats["dist_non_events"] = (stats["non_events"] + self.smoothing) / (total_non_events + self.smoothing)
            stats["woe"] = np.log(stats["dist_events"] / stats["dist_non_events"])

            self.woe_maps[col] = stats["woe"].to_dict()

        return self

    def transform(self, X):
        X = X.copy()
        for col, woe_map in self.woe_maps.items():
            if col in X.columns:
                X[col] = X[col].map(woe_map).fillna(0)  # unseen → 0 (neutral)
        return X

In [22]:
class CorrelationFilter(BaseEstimator, TransformerMixin):
    def fit(self, X, y=None):
        X = pd.DataFrame(X).copy()
        # Sample if too large
        if len(X) > 5000:
            X_sample = X.sample(5000, random_state=42)
        else:
            X_sample = X
            
        corr_matrix = X_sample.corr().abs()  # faster on sample
        upper = corr_matrix.where(
            np.triu(np.ones(corr_matrix.shape), k=1).astype(bool)
        )
        self.cols_to_drop_ = [col for col in upper.columns if any(upper[col] > 0.8)]
        print(self.cols_to_drop_)
        return self

    def transform(self, X):
        return pd.DataFrame(X).drop(columns=self.cols_to_drop_, errors='ignore')

# Feature Selection

In [32]:
import numpy as np
import pandas as pd
import shap

from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.pipeline import Pipeline
from xgboost import XGBClassifier


class SHAPFeatureSelector(BaseEstimator, TransformerMixin):
    
    def __init__(
        self,
        sample_size=5000,
        drop_percent=0.2,
        random_state=42
    ):
        self.sample_size = sample_size
        self.drop_percent = drop_percent
        self.random_state = random_state
        
    def fit(self, X, y):
        
        # sample for speed
        n = min(self.sample_size, len(X))
        
        rng = np.random.RandomState(self.random_state)
        idx = rng.choice(len(X), n, replace=False)

        X_sample = X.iloc[idx]
        y_sample = y.iloc[idx] if hasattr(y, "iloc") else y[idx]
        
        # internal xgboost model for feature ranking
        model = XGBClassifier(
            device="cuda",
            tree_method="hist",
            eval_metric="auc",
            n_estimators=300,
            max_depth=4,
            learning_rate=0.1,
            random_state=self.random_state, 
            enable_categorical=True,
        )

        model.fit(X_sample, y_sample)

        # SHAP
        explainer = shap.TreeExplainer(model)
        shap_values = explainer.shap_values(X_sample)

        importance = pd.Series(
            np.abs(shap_values).mean(axis=0),
            index=X.columns
        ).sort_values(ascending=False)

        threshold = importance.quantile(self.drop_percent)

        self.selected_columns_ = importance[
            importance > threshold
        ].index.tolist()

        print(
            f"Selected {len(self.selected_columns_)} "
            f"out of {X.shape[1]} features"
        )

        return self

    def transform(self, X):
        return X[self.selected_columns_]



In [36]:
params_to_try = [
    {"n_estimators": 600, "max_depth": 20, "min_samples_split": 5,  "min_samples_leaf": 2, "max_features": "sqrt"},
    {"n_estimators": 600, "max_depth": 15, "min_samples_split": 5,  "min_samples_leaf": 2, "max_features": "sqrt"},
    {"n_estimators": 800, "max_depth": 10, "min_samples_split": 10, "min_samples_leaf": 4, "max_features": "sqrt"},
    {"n_estimators": 1000, "max_depth": 20, "min_samples_split": 2,  "min_samples_leaf": 1, "max_features": "sqrt"},
    {"n_estimators": 800, "max_depth": 30, "min_samples_split": 5,  "min_samples_leaf": 2, "max_features": "log2"},
]

for params in params_to_try:
    pipeline_rf.set_params(**{f"model__{k}": v for k, v in params.items()})
    run_experiment_rf(pipeline_rf, f"RF_{params}_woe_feature_selection", "Random Forest", X_train, X_val, y_train, y_val)

Selected 211 out of 433 features


2026/05/05 08:58:48 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/05/05 08:58:55 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


🏃 View run RF_{'n_estimators': 600, 'max_depth': 20, 'min_samples_split': 5, 'min_samples_leaf': 2, 'max_features': 'sqrt'}_woe_feature_selection at: https://dagshub.com/icosahedron31/IEEE-CIS-Fraud-Detection.mlflow/#/experiments/8/runs/90495bfb85d0491f9f57241d6755fbb1
🧪 View experiment at: https://dagshub.com/icosahedron31/IEEE-CIS-Fraud-Detection.mlflow/#/experiments/8
Selected 211 out of 433 features


2026/05/05 09:09:42 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/05/05 09:09:49 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


🏃 View run RF_{'n_estimators': 600, 'max_depth': 15, 'min_samples_split': 5, 'min_samples_leaf': 2, 'max_features': 'sqrt'}_woe_feature_selection at: https://dagshub.com/icosahedron31/IEEE-CIS-Fraud-Detection.mlflow/#/experiments/8/runs/bf51817e69ee48df8c4d251484a1c07e
🧪 View experiment at: https://dagshub.com/icosahedron31/IEEE-CIS-Fraud-Detection.mlflow/#/experiments/8
Selected 211 out of 433 features


...

🏃 View run RF_{'n_estimators': 800, 'max_depth': 10, 'min_samples_split': 10, 'min_samples_leaf': 4, 'max_features': 'sqrt'}_woe_feature_selection at: https://dagshub.com/icosahedron31/IEEE-CIS-Fraud-Detection.mlflow/#/experiments/8/runs/a671dabe6a294f41ae0acae54e52077e
🧪 View experiment at: https://dagshub.com/icosahedron31/IEEE-CIS-Fraud-Detection.mlflow/#/experiments/8
Selected 211 out of 433 features
🏃 View run RF_{'n_estimators': 1000, 'max_depth': 20, 'min_samples_split': 2, 'min_samples_leaf': 1, 'max_features': 'sqrt'}_woe_feature_selection at: https://dagshub.com/icosahedron31/IEEE-CIS-Fraud-Detection.mlflow/#/experiments/8/runs/37927083b4344f399120280165937495
🧪 View experiment at: https://dagshub.com/icosahedron31/IEEE-CIS-Fraud-Detection.mlflow/#/experiments/8


KeyboardInterrupt: 

<Figure size 640x480 with 0 Axes>

<Figure size 640x480 with 0 Axes>

<Figure size 640x480 with 0 Axes>

In [35]:
from sklearn.ensemble import RandomForestClassifier
woe_cols = X_train.select_dtypes(include=["object", "category"]).columns.tolist()


pipeline_rf = Pipeline([
    ("woe", WOEEncoder()),
    ("featureSelection", SHAPFeatureSelector()),
    
    ("model", RandomForestClassifier(
        n_estimators=1000,
        max_depth=10,
        min_samples_split=5,
        min_samples_leaf=2,
        max_features="sqrt",
   
        class_weight="balanced",
        n_jobs=-1,
        random_state=42, 
       
    ))
])

smote

In [53]:
import pandas as pd
from imblearn.pipeline import Pipeline  # ← imblearn, not sklearn
from sklearn.compose import ColumnTransformer, make_column_selector
from sklearn.preprocessing import OrdinalEncoder
from imblearn.over_sampling import SMOTE
from xgboost import XGBClassifier

encoder = ColumnTransformer(
    transformers=[
        (
            "cat",
            OrdinalEncoder(
                handle_unknown="use_encoded_value",
                unknown_value=-1
            ),
            make_column_selector(dtype_include=["object", "category"])
        )
    ],
    remainder="passthrough"
)

pipeline = Pipeline([
    ("preprocess", FraudPreprocessor()),
    ("encode", encoder),
    ("smote", SMOTE(random_state=42)),
    ("model", XGBClassifier(
        device="cuda",
        scale_pos_weight=scale_pos_weight,
        n_estimators=300,
        learning_rate=0.1,
        max_depth=4,
        subsample=0.8,
        colsample_bytree=0.8,
        eval_metric="auc",
        tree_method="hist",
        random_state=42
    ))
])

In [54]:
params_to_try = [
    {"n_estimators": 300, "max_depth": 10, "min_samples_split": 5,  "min_samples_leaf": 2, "max_features": "sqrt"},
    {"n_estimators": 300, "max_depth": 15, "min_samples_split": 5,  "min_samples_leaf": 2, "max_features": "sqrt"},
    {"n_estimators": 500, "max_depth": 10, "min_samples_split": 10, "min_samples_leaf": 4, "max_features": "sqrt"},
    {"n_estimators": 500, "max_depth": 20, "min_samples_split": 2,  "min_samples_leaf": 1, "max_features": "sqrt"},
    {"n_estimators": 300, "max_depth": 30, "min_samples_split": 5,  "min_samples_leaf": 2, "max_features": "log2"},
]

for params in params_to_try:
    pipeline_rf.set_params(**{f"model__{k}": v for k, v in params.items()})
    run_experiment_rf(pipeline_rf, f"RF_{params}_woe", "Random Forest", X_train, X_val, y_train, y_val)

2026/05/04 09:29:22 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/05/04 09:29:29 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


🏃 View run XGBoost_{'n_estimators': 200, 'learning_rate': 0.1, 'max_depth': 4} at: https://dagshub.com/icosahedron31/IEEE-CIS-Fraud-Detection.mlflow/#/experiments/7/runs/ff8e9b182f6744a4b483191ccdcb3be6
🧪 View experiment at: https://dagshub.com/icosahedron31/IEEE-CIS-Fraud-Detection.mlflow/#/experiments/7


2026/05/04 09:31:07 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/05/04 09:31:14 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


🏃 View run XGBoost_{'n_estimators': 300, 'learning_rate': 0.05, 'max_depth': 6} at: https://dagshub.com/icosahedron31/IEEE-CIS-Fraud-Detection.mlflow/#/experiments/7/runs/0b91fd8c8e0545edaba0c1d7498fec24
🧪 View experiment at: https://dagshub.com/icosahedron31/IEEE-CIS-Fraud-Detection.mlflow/#/experiments/7


2026/05/04 09:33:09 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/05/04 09:33:16 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


🏃 View run XGBoost_{'n_estimators': 500, 'learning_rate': 0.01, 'max_depth': 6} at: https://dagshub.com/icosahedron31/IEEE-CIS-Fraud-Detection.mlflow/#/experiments/7/runs/11841f6169124d21a786864079f060c1
🧪 View experiment at: https://dagshub.com/icosahedron31/IEEE-CIS-Fraud-Detection.mlflow/#/experiments/7


<Figure size 640x480 with 0 Axes>

<Figure size 640x480 with 0 Axes>

<Figure size 640x480 with 0 Axes>

Undersampling

# Feature Engineering 

In [15]:
import pandas as pd
import numpy as np
from sklearn.base import BaseEstimator, TransformerMixin


class FeatureEngineer(BaseEstimator, TransformerMixin):

    def __init__(self):
        self.email_freqs_ = {}
        self.fraud_rates_ = {}

    def fit(self, X, y=None):
        X = X.copy()

        for col in ['P_emaildomain', 'R_emaildomain']:
            if col in X.columns:
                self.email_freqs_[col] = (
                    X[col]
                    .value_counts()
                    .to_dict()
                )

        if y is not None:
            tmp = X.copy()
            tmp['_target'] = y.values

            for col in ['P_emaildomain', 'R_emaildomain']:
                if col in X.columns:
                    self.fraud_rates_[col] = (
                        tmp.groupby(col)['_target']
                        .mean()
                        .to_dict()
                    )

        return self


    def transform(self, X):
        X = X.copy()
        original_index = X.index

        # Amount features
        X['amt_log'] = np.log1p(X['TransactionAmt'])
        X['amt_rounded'] = (
            X['TransactionAmt'] % 1 == 0
        ).astype(int)

        # Time features
        X['hour'] = (
            X['TransactionDT'] // 3600
        ) % 24

        X['day_of_week'] = (
            X['TransactionDT'] // (3600 * 24)
        ) % 7

        X['is_weekend'] = (
            X['day_of_week'].isin([5, 6])
        ).astype(int)

        # Email features
        if {'P_emaildomain', 'R_emaildomain'}.issubset(X.columns):
            X['email_match'] = (
                X['P_emaildomain']
                == X['R_emaildomain']
            ).astype(int)

        for col, freq_map in self.email_freqs_.items():
            X[f'{col}_freq'] = X[col].map(freq_map)

        # Target encoding
        for col, fraud_map in self.fraud_rates_.items():
            X[f'{col}_fraud_rate'] = X[col].map(fraud_map)

        # Time since previous transaction
        if {'card1', 'TransactionDT'}.issubset(X.columns):

            tmp = X[['card1', 'TransactionDT']].copy()
            tmp = tmp.sort_values('TransactionDT')

            tmp['time_since_last_txn'] = (
                tmp.groupby('card1')['TransactionDT']
                .diff()
            )

            X['time_since_last_txn'] = (
                tmp['time_since_last_txn']
                .reindex(original_index)
            )

        return X

In [16]:
from sklearn.model_selection import train_test_split



X_train_eng, X_val_eng, y_train, y_val = train_test_split(
    X_engineered, y,
    test_size=0.2,
    random_state=42
)

X_train_eng.shape, X_val_eng.shape


((472432, 467), (118108, 467))

In [33]:
import pandas as pd
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer, make_column_selector
from sklearn.preprocessing import OrdinalEncoder
from sklearn.ensemble import RandomForestClassifier


woe_cols = X_train_eng.select_dtypes(include=["object", "category"]).columns.tolist()
X_train_eng = X_train_eng.astype({c: "object" for c in X_train_eng.select_dtypes("category").columns})
X_val_eng   = X_val_eng.astype({c: "object" for c in X_val.select_dtypes("category").columns})

pipeline_rf = Pipeline([
    ("engineer", FeatureEngineer()),
    ("missing", MissingValueFilter()),
    ("woe", WOEEncoder()),
    ("Correlation", CorrelationFilter()),
    ("model", RandomForestClassifier(
        n_estimators=1000,
        max_depth=10,
        min_samples_split=5,
        min_samples_leaf=2,
        max_features="sqrt",
        class_weight="balanced",
        n_jobs=-1,
        random_state=42
    ))
])

In [34]:
params_to_try = [
    {"n_estimators": 600, "max_depth": 10, "min_samples_split": 5,  "min_samples_leaf": 2, "max_features": "sqrt"},
    {"n_estimators": 800, "max_depth": 15, "min_samples_split": 5,  "min_samples_leaf": 2, "max_features": "sqrt"},
    {"n_estimators": 1000, "max_depth": 10, "min_samples_split": 10, "min_samples_leaf": 4, "max_features": "sqrt"},
    {"n_estimators": 600, "max_depth": 20, "min_samples_split": 2,  "min_samples_leaf": 1, "max_features": "sqrt"},
    {"n_estimators": 500, "max_depth": 30, "min_samples_split": 5,  "min_samples_leaf": 2, "max_features": "log2"},
]

for params in params_to_try:
    pipeline_rf.set_params(**{f"model__{k}": v for k, v in params.items()})
    run_experiment_rf(pipeline_rf, f"RF_{params}_feature_eng_woe_and_heavy_cleaning", "Random Forest", X_train, X_val, y_train, y_val)

['TransactionDT', 'C2', 'C4', 'C6', 'C7', 'C8', 'C9', 'C10', 'C11', 'C12', 'C13', 'C14', 'D2', 'D6', 'D12', 'M2', 'M3', 'M8', 'M9', 'V5', 'V9', 'V11', 'V13', 'V15', 'V16', 'V17', 'V18', 'V20', 'V21', 'V22', 'V26', 'V28', 'V29', 'V30', 'V31', 'V32', 'V33', 'V34', 'V36', 'V38', 'V39', 'V40', 'V42', 'V43', 'V44', 'V45', 'V47', 'V48', 'V49', 'V50', 'V51', 'V52', 'V54', 'V57', 'V58', 'V59', 'V60', 'V62', 'V63', 'V64', 'V67', 'V69', 'V70', 'V71', 'V72', 'V73', 'V74', 'V76', 'V79', 'V80', 'V81', 'V83', 'V84', 'V85', 'V87', 'V90', 'V91', 'V92', 'V93', 'V94', 'V96', 'V97', 'V100', 'V101', 'V102', 'V103', 'V104', 'V105', 'V106', 'V110', 'V112', 'V113', 'V114', 'V119', 'V126', 'V127', 'V128', 'V132', 'V133', 'V134', 'V136', 'V137', 'V138', 'V140', 'V141', 'V142', 'V143', 'V145', 'V147', 'V149', 'V150', 'V151', 'V152', 'V153', 'V154', 'V155', 'V156', 'V157', 'V158', 'V159', 'V160', 'V161', 'V162', 'V163', 'V164', 'V165', 'V166', 'V167', 'V168', 'V175', 'V177', 'V178', 'V179', 'V180', 'V181', 'V182

2026/05/04 18:23:29 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/05/04 18:23:36 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


🏃 View run RF_{'n_estimators': 300, 'max_depth': 10, 'min_samples_split': 5, 'min_samples_leaf': 2, 'max_features': 'sqrt'}_feature_eng_woe_and_heavy_cleaning at: https://dagshub.com/icosahedron31/IEEE-CIS-Fraud-Detection.mlflow/#/experiments/8/runs/3396c5ad2b4f43b997bf44b13bb8f9dd
🧪 View experiment at: https://dagshub.com/icosahedron31/IEEE-CIS-Fraud-Detection.mlflow/#/experiments/8
['TransactionDT', 'C2', 'C4', 'C6', 'C7', 'C8', 'C9', 'C10', 'C11', 'C12', 'C13', 'C14', 'D2', 'D6', 'D12', 'M2', 'M3', 'M8', 'M9', 'V5', 'V9', 'V11', 'V13', 'V15', 'V16', 'V17', 'V18', 'V20', 'V21', 'V22', 'V26', 'V28', 'V29', 'V30', 'V31', 'V32', 'V33', 'V34', 'V36', 'V38', 'V39', 'V40', 'V42', 'V43', 'V44', 'V45', 'V47', 'V48', 'V49', 'V50', 'V51', 'V52', 'V54', 'V57', 'V58', 'V59', 'V60', 'V62', 'V63', 'V64', 'V67', 'V69', 'V70', 'V71', 'V72', 'V73', 'V74', 'V76', 'V79', 'V80', 'V81', 'V83', 'V84', 'V85', 'V87', 'V90', 'V91', 'V92', 'V93', 'V94', 'V96', 'V97', 'V100', 'V101', 'V102', 'V103', 'V104', 'V

2026/05/04 18:28:03 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/05/04 18:28:10 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


🏃 View run RF_{'n_estimators': 300, 'max_depth': 15, 'min_samples_split': 5, 'min_samples_leaf': 2, 'max_features': 'sqrt'}_feature_eng_woe_and_heavy_cleaning at: https://dagshub.com/icosahedron31/IEEE-CIS-Fraud-Detection.mlflow/#/experiments/8/runs/9aaea561724342209fca9a322fa00482
🧪 View experiment at: https://dagshub.com/icosahedron31/IEEE-CIS-Fraud-Detection.mlflow/#/experiments/8
['TransactionDT', 'C2', 'C4', 'C6', 'C7', 'C8', 'C9', 'C10', 'C11', 'C12', 'C13', 'C14', 'D2', 'D6', 'D12', 'M2', 'M3', 'M8', 'M9', 'V5', 'V9', 'V11', 'V13', 'V15', 'V16', 'V17', 'V18', 'V20', 'V21', 'V22', 'V26', 'V28', 'V29', 'V30', 'V31', 'V32', 'V33', 'V34', 'V36', 'V38', 'V39', 'V40', 'V42', 'V43', 'V44', 'V45', 'V47', 'V48', 'V49', 'V50', 'V51', 'V52', 'V54', 'V57', 'V58', 'V59', 'V60', 'V62', 'V63', 'V64', 'V67', 'V69', 'V70', 'V71', 'V72', 'V73', 'V74', 'V76', 'V79', 'V80', 'V81', 'V83', 'V84', 'V85', 'V87', 'V90', 'V91', 'V92', 'V93', 'V94', 'V96', 'V97', 'V100', 'V101', 'V102', 'V103', 'V104', 'V

...

<Figure size 640x480 with 0 Axes>

<Figure size 640x480 with 0 Axes>

<Figure size 640x480 with 0 Axes>

<Figure size 640x480 with 0 Axes>

<Figure size 640x480 with 0 Axes>